# Contextual neighbor sampler tests

Tests for the current, edge-ID-based `ContextualNeighborSampler` API.

In [1]:
import numpy as np

from utils.ContextualNeighborSampler import (
    ContextualNeighborSampler,
    HyperParameters,
    get_contextual_neighbor_sampler,
)
from utils.DataLoader import Data, get_link_prediction_data
from utils.utils import get_neighbor_sampler

## Synthetic graph

In [2]:
eps, gamma, N = 0.5, 2, 3
hp = HyperParameters(eps=eps, number_of_neighbors=N, gamma=gamma)

# (student, question, edge_id, timestamp); edge IDs are deliberately dense.
interactions = [
    (1, 10, 1, 1.0),
    (1, 11, 2, 2.0),
    (1, 12, 3, 3.0),
    (1, 13, 4, 4.0),
    (1, 14, 5, 5.0),
    (1, 15, 6, 6.0),
]
src, dst, edge_ids, timestamps = map(np.asarray, zip(*interactions))
synthetic_data = Data(
    src_node_ids=src.astype(np.int64),
    dst_node_ids=dst.astype(np.int64),
    node_interact_times=timestamps.astype(np.float64),
    edge_ids=edge_ids.astype(np.int64),
    labels=np.ones(len(interactions), dtype=np.int64),
)

node_features_synthetic = np.zeros((16, 1), dtype=np.int64)
node_features_synthetic[[10, 12, 14], 0] = 2
node_features_synthetic[[11, 13, 15], 0] = 3
sampler = get_contextual_neighbor_sampler(
    data=synthetic_data, hp=hp, node_raw_features=node_features_synthetic, seed=42
)

## Contextual ranking

In [3]:
# Query edge 6: its strict history consists of edges 1..5.
neighbors, edges, times = sampler.get_neighbors_dst(
    node_ids=np.array([1]),
    target_skill=np.array([2]),
    node_interact_times=np.array([6.0]),
    edge_ids=np.array([6]),
)
neighbors, edges, times = (x.cpu().numpy() for x in (neighbors, edges, times))
selected_skills = sampler.node_skills[neighbors[0]]

print("neighbors:", neighbors)
print("edges:", edges)
print("times:", times)
print("skills:", selected_skills)

assert neighbors.shape == (1, N)
assert np.all(np.diff(times[0]) >= 0)
assert np.all(times[times > 0] < 6.0)
assert np.count_nonzero(selected_skills == 2) >= 2

neighbors: [[12 13 14]]
edges: [[3 4 5]]
times: [[3. 4. 5.]]
skills: [2 3 2]


## No temporal leakage

In [4]:
# Query edge 4 must exclude edge 4 itself and all later interactions.
_, _, earlier_times = sampler.get_neighbors_dst(
    node_ids=np.array([1]),
    target_skill=np.array([3]),
    node_interact_times=np.array([4.0]),
    edge_ids=np.array([4]),
)
earlier_times = earlier_times.cpu().numpy()
print("times before edge 4:", earlier_times)
assert np.all(earlier_times[earlier_times > 0] < 4.0)

times before edge 4: [[1. 2. 3.]]


## Comparison on Assistments 2017

In [2]:
dataset_name = "assist12"
node_raw_features, _, full_data, train_data, *_ = get_link_prediction_data(
    dataset_name=dataset_name, val_ratio=0.1, test_ratio=0.1
)

The dataset has 6001504 interactions, involving 148331 different nodes
The training dataset has 3706249 interactions, involving 121634 different nodes
The validation dataset has 600148 interactions, involving 52570 different nodes
The test dataset has 600151 interactions, involving 53359 different nodes
The new node validation dataset has 348618 interactions, involving 41319 different nodes
The new node test dataset has 499900 interactions, involving 47552 different nodes
14833 nodes were used for the inductive testing, i.e. are never seen during training


In [ ]:
real_hp = HyperParameters(eps=0.5, number_of_neighbors=50, gamma=15)
contextual = get_contextual_neighbor_sampler(
    data=full_data, hp=real_hp, node_raw_features=node_raw_features, seed=42
)
traditional = get_neighbor_sampler(full_data, sample_neighbor_strategy="recent", seed=42)

In [25]:
rng = np.random.default_rng(77)
sample_size = min(1000, full_data.num_interactions)
idx = rng.choice(full_data.num_interactions, size=sample_size, replace=False)
node_ids = full_data.src_node_ids[idx]
question_ids = full_data.dst_node_ids[idx]
query_times = full_data.node_interact_times[idx]
query_edges = full_data.edge_ids[idx]
target_skills = node_raw_features[question_ids, 0].astype(np.int64)

context_nodes, _, _ = contextual.get_neighbors_dst(
    node_ids=node_ids, target_skill=target_skills,
    node_interact_times=query_times, edge_ids=query_edges,
)
context_nodes = context_nodes.cpu().numpy()
baseline_nodes, _, _ = traditional.get_historical_neighbors(
    node_ids=node_ids, node_interact_times=query_times, num_neighbors=real_hp.N
)

def same_skill_ratio(neighbors):
    valid = neighbors != 0
    matches = node_raw_features[neighbors, 0] == target_skills[:, None]
    return matches[valid].mean() if valid.any() else 0.0

print("Contextual same-skill ratio:", same_skill_ratio(context_nodes))
print("Traditional same-skill ratio:", same_skill_ratio(baseline_nodes))

Contextual same-skill ratio: 0.5655870540432628
Traditional same-skill ratio: 0.532869859313714
